In [ ]:
import cv2
import ale_py
import gymnasium as gym
import torch

from stable_baselines3 import DQN, PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecTransposeImage, VecFrameStack
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.atari_wrappers import FireResetEnv

In [2]:
import os
import time
import glob
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

In [3]:
gym.register_envs(ale_py)

In [4]:
class WhitePatchBreakout(gym.Wrapper):
    def __init__(self, env, apply_patch=False, region=(20, 40, 30, 30)):
        super().__init__(env)
        self.apply_patch = apply_patch
        self.x, self.y, self.w, self.h = region

    def _paint_patch(self, img):
        if not self.apply_patch:
            return img
        
        # Copia para não alterar o buffer original inadvertidamente
        img = img.copy()
        
        # Detecta dimensões (pode ser 84x84 ou a resolução nativa no render)
        H, W = img.shape[:2]
        
        x, y, w, h = self.x, self.y, self.w, self.h
        
        # Garante que o desenho não exceda as bordas
        y_end = min(y + h, H)
        x_end = min(x + w, W)
        
        if y < y_end and x < x_end:
            # Pinta de branco (255). Funciona tanto para RGB quanto Grayscale se ajustado
            img[y:y_end, x:x_end] = 255
            
        return img

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        return self._paint_patch(obs), info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        return self._paint_patch(obs), reward, terminated, truncated, info

    def render(self):
        # Mágica para o vídeo: Pega o render original e aplica a mancha nele
        img = self.env.render()
        return self._paint_patch(img)

In [5]:
class BlurBreakout(gym.Wrapper):
    def __init__(self, env, apply_blur=False, region=(20, 40, 30, 30), ksize=21):
        super().__init__(env)
        self.apply_blur = apply_blur
        self.x, self.y, self.w, self.h = region
        self.ksize = ksize if ksize % 2 == 1 else ksize + 1

    def _apply_blur(self, img):
        if not self.apply_blur:
            return img
        
        img = img.copy()
        H, W = img.shape[:2]
        x, y, w, h = self.x, self.y, self.w, self.h
        
        y_end = min(y + h, H)
        x_end = min(x + w, W)
        
        if y < y_end and x < x_end:
            roi = img[y:y_end, x:x_end]
            # Aplica Gaussian Blur na região de interesse
            blurred_roi = cv2.GaussianBlur(roi, (self.ksize, self.ksize), 0)
            img[y:y_end, x:x_end] = blurred_roi
            
        return img

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        return self._apply_blur(obs), info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        return self._apply_blur(obs), reward, terminated, truncated, info
    
    def render(self):
        img = self.env.render()
        return self._apply_blur(img)

In [6]:
class GrayscaleWrapper(gym.Wrapper):
    def __init__(self, env):
        super().__init__(env)
        # Força explicitamente o shape (H, W, 1) para compatibilidade com SB3
        old_shape = self.observation_space.shape
        self.observation_space = gym.spaces.Box(
            low=0, high=255, shape=(old_shape[0], old_shape[1], 1), dtype=np.uint8
        )

    def _convert(self, obs):
        # Se for RGB (3 canais), converte para Gray e mantém dimensão extra
        if obs.ndim == 3 and obs.shape[2] == 3:
            return cv2.cvtColor(obs, cv2.COLOR_RGB2GRAY)[:, :, None]
        return obs

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        return self._convert(obs), info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        return self._convert(obs), reward, terminated, truncated, info

In [7]:
def make_training_env(with_blur=False, change='white_patch', zone='middle', log_dir="./logs/"):
    """
    Cria ambiente com zonas configuráveis.
    Zonas disponíveis: 'top', 'middle', 'bottom'.
    """
    if log_dir:
        os.makedirs(log_dir, exist_ok=True)

    # --- CORREÇÃO DE ESCALA ---
    # A imagem da IA tem apenas 84x84 pixels.
    # Tamanho da Mancha: 40x40 pixels.
    # Centralização X: (84 - 40) / 2 = 22.
    
    presets = {
        # ZONA 1: TOP (TIJOLOS)
        # Y=25: Centraliza a mancha na massa de tijolos coloridos.
        'top':    (32, 25, 20, 20), 

        # ZONA 2: MIDDLE (TRAJETÓRIA)
        # Y=50: Fica no espaço vazio logo acima da linha de perigo.
        'middle': (32, 50, 20, 20), 

        # ZONA 3: BOTTOM (PADDLE)
        # Y=64: Vai exatamente até o pixel 84 (64+20=84).
        # Cobre apenas o centro do paddle, exigindo precisão.
        'bottom': (32, 64, 20, 20)  
    }
    
    region = presets.get(zone, (22, 38, 40, 40))

    def _init():
        env = gym.make("ALE/Breakout-v5", render_mode="rgb_array")
        env = FireResetEnv(env)
        
        # 1. Redimensiona PRIMEIRO (O mundo vira 84x84 aqui)
        env = gym.wrappers.ResizeObservation(env, (84, 84))
        
        # 2. Aplica a Mancha DEPOIS (Usando as coordenadas convertidas acima)
        if change == 'white_patch':
            env = WhitePatchBreakout(env, apply_patch=with_blur, region=region)
        elif change == 'blur':
            env = BlurBreakout(env, apply_blur=with_blur, region=region, ksize=21)
        
        env = GrayscaleWrapper(env)
        
        if log_dir:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = os.path.join(log_dir, f"monitor_{zone}_{timestamp}")
            env = Monitor(env, filename)
            
        return env

    env = DummyVecEnv([_init])
    env = VecTransposeImage(env)
    env = VecFrameStack(env, n_stack=4)
    
    return env

In [25]:
def compute_saliency(model, obs):
    """
    Calcula o mapa de saliência (atenção) via Gradiente.
    Versão robusta para DQN e PPO.
    """
    # 1. Prepara a observação para o PyTorch
    # Garante que está no formato (Batch, Channels, H, W) e no device correto (CPU/GPU)
    device = model.device
    obs_tensor = torch.from_numpy(obs).float().to(device)
    obs_tensor.requires_grad_() # Liga o rastreamento de gradiente

    # Coloca o modelo em modo de avaliação (desliga Dropout/BatchNorm se houver)
    model.policy.set_training_mode(False)

    # 2. Forward Pass (Calcula o Score)
    if isinstance(model, DQN):
        # --- DQN ---
        # Acessa diretamente a q_net. Ela já engloba a extração de features + MLP
        # O retorno são os Q-Values brutos para todas as ações
        q_values = model.q_net(obs_tensor)
        
        # Queremos explicar a ação que o agente ESCOLHEU (o maior Q-Value)
        target_action = torch.argmax(q_values)
        score = q_values[0, target_action]

    elif isinstance(model, PPO):
        # --- PPO ---
        # O PPO usa uma distribuição de probabilidade
        # distribution = model.policy.get_distribution(obs_tensor)
        
        # Precisamos passar pela rede de features primeiro, depois pela rede de ação (Actor)
        features = model.policy.extract_features(obs_tensor)
        latent_pi = model.policy.mlp_extractor.forward_actor(features)
        action_logits = model.policy.action_net(latent_pi)
        
        # Pega a ação mais provável e seu valor (logit)
        target_action = torch.argmax(action_logits)
        score = action_logits[0, target_action]

    else:
        raise ValueError("Modelo não suportado. Use DQN ou PPO.")

    # 3. Backward Pass (Calcula o Gradiente)
    # Zera gradientes anteriores para não acumular sujeira
    model.policy.zero_grad()
    
    # Calcula a derivada do Score em relação aos Pixels de entrada
    score.backward()

    # 4. Processa o Gradiente para virar uma Imagem
    # Saliency = Valor absoluto do gradiente nos pixels
    # Pegamos o gradiente do canal 3 (último frame empilhado, o 'agora')
    # obs_tensor.grad tem shape (1, 4, 84, 84)
    gradients = obs_tensor.grad.data.abs().cpu().numpy()
    saliency = gradients[0, 3, :, :] 

    # Normalização Robusta (para o mapa ficar bem visível)
    # Evita divisão por zero e estica o contraste
    val_min = saliency.min()
    val_max = saliency.max()
    if val_max > val_min:
        saliency = (saliency - val_min) / (val_max - val_min)
    
    return saliency

def visualize_attention(model, env, title="Attention Map"):
    # Reseta e avança um pouco para ter uma cena de jogo real
    obs = env.reset()
    for _ in range(30):
        # Usa o modelo para jogar um pouco
        action, _ = model.predict(obs, deterministic=True)
        # Se o jogo acabar no meio, reseta
        obs, _, done, _ = env.step(action)
        if isinstance(done, list): done = done[0] # Tratamento para VecEnv
        if done: obs = env.reset()

    # Calcula Saliência
    saliency_map = compute_saliency(model, obs)
    
    # Pega a imagem original (Frame 3 do Stack) para fundo
    # obs shape: (1, 4, 84, 84)
    original_img = obs[0, 3, :, :]

    # --- PLOTAGEM ---
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    # 1. Visão do Agente (O que ele vê: Cinza 84x84)
    axes[0].imshow(original_img, cmap='gray')
    axes[0].set_title("Visão do Agente (Grayscale)")
    axes[0].axis('off')
    
    # 2. Mapa de Atenção (Onde ele foca)
    # Plotamos a imagem original escura e o mapa de calor por cima
    axes[1].imshow(original_img, cmap='gray', alpha=0.5)
    
    # 'jet', 'hot' ou 'inferno' são bons colormaps para calor
    im = axes[1].imshow(saliency_map, cmap='inferno', alpha=0.8) 
    
    axes[1].set_title(f"Saliency Map (Foco de Atenção)\n{title}")
    axes[1].axis('off')
    
    # Adiciona barra de cor lateral
    cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
    cbar.set_label('Intensidade do Gradiente', rotation=270, labelpad=15)
    
    plt.tight_layout()
    plt.show()

In [ ]:
agente = 'normal' 
blur = False    
zone = 'top'   

path = f"treinos/treino_ppo/ppo_breakout_{agente}"
if not os.path.exists(path) and os.path.exists(path + ".zip"):
    path += ".zip"
print(f"Carregando: {path}")
model = PPO.load(path)
env = make_training_env(with_blur=blur, zone=zone, log_dir=None)
visualize_attention(model, env, title=f"PPO {agente} no Ambiente")
env.close()

Carregando: treino_dqn/dqn_breakout_normal.zip


MemoryError: Unable to allocate 2.63 GiB for an array with shape (100000, 1, 4, 84, 84) and data type uint8

In [ ]:
agente = 'normal' 
blur = False    
zone = 'top'   

path = f"logs\ppo\BreakoutNoFrameskip-v4_1\BreakoutNoFrameskip-v4"
if not os.path.exists(path) and os.path.exists(path + ".zip"):
    path += ".zip"
print(f"Carregando: {path}")
model = PPO.load(path)
env = make_training_env(with_blur=blur, zone=zone, log_dir=None)
visualize_attention(model, env, title=f"PPO pré-treinado no Ambiente")
env.close()